آگهی های دارای قیمت سال ساخت و یا مساحت غیر عادی را شناسایی کنید

In [2]:
import re
import numpy as np
import pandas as pd
from pathlib import Path

<div dir="rtl" align="right">

## Outlier Policy — Numeric Normalization, Flagging and Audit
<div dir="rtl" align="right">
در این مرحله بدون حذف هیچ رکوردی، ستون‌های عددی موردنیاز برای شناسایی داده‌های پرت آماده می‌شوند.
<div dir="rtl" align="right">  
قواعد اصلی:

- ستون‌های خام تغییر نمی‌کنند.
- برای تحلیل Outlier از ستون‌های استانداردشده استفاده می‌شود.
- قیمت فروش، رهن و اجاره بر اساس رژیم قیمت جدا بررسی می‌شوند.
- مساحت و سال ساخت جداگانه Flag می‌شوند.
- هیچ داده‌ای حذف نمی‌شود؛ فقط Flag و Reason ثبت می‌شود.
- خروجی‌های Audit برای گزارش و Power BI ذخیره می‌شوند.

<div dir="rtl" align="right">
</div>

In [4]:
df = pd.read_feather("../Outputs/02_df.feather")

In [ ]:
# ==============================
# Outlier Policy — Unified Safe Final Block
# ==============================



# BASE_DIR = Path(r"C:\Users\Abbas\Desktop\housing_market_project")
# OUTPUT_TABLES = BASE_DIR / "outputs" / "tables"
# OUTPUT_TABLES.mkdir(parents=True, exist_ok=True)

# ------------------------------
# 1) Safety Check
# ------------------------------
required_cols = ["price_regime", "price_status"]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns for Outlier Policy: {missing_cols}")
outlier_df = df.copy()
# ------------------------------
# 2) Remove Old Outlier Columns
# ------------------------------
old_patterns = [
    r"^sale_price_num$",
    r"^deposit_amount_num$",
    r"^monthly_rent_num$",
    r"^building_size_num$",
    r"^construction_year_num$",
    r"^outlier_",
    r"^has_any_outlier_flag$",
    r"^group_count",
    r"^lower_bound",
    r"^upper_bound",
    r"^price_iqr_",
]
old_cols = [
    col for col in outlier_df.columns
    if any(re.match(pattern, str(col)) for pattern in old_patterns)
]
outlier_df = outlier_df.drop(columns=old_cols, errors="ignore")
# ------------------------------
# 3) Safe Numeric Normalization
# ------------------------------
digit_map = str.maketrans(
    "۰۱۲۳۴۵۶۷۸۹٠١٢٣٤٥٦٧٨٩",
    "01234567890123456789"
)
def safe_numeric(series):
    return (
        series
        .astype(str)
        .str.translate(digit_map)
        .str.replace("٬", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.replace(" ", "", regex=False)
        .str.replace(r"[^\d\.\-]", "", regex=True)
        .replace(["", "nan", "None", "NaN"], np.nan)
        .pipe(pd.to_numeric, errors="coerce")
    )
for col in ["sale_price", "deposit_amount", "monthly_rent"]:
    if col in outlier_df.columns:
        outlier_df[f"{col}_num"] = safe_numeric(outlier_df[col])
    else:
        outlier_df[f"{col}_num"] = np.nan
if "building_size" in outlier_df.columns:
    outlier_df["building_size_num"] = safe_numeric(outlier_df["building_size"])
else:
    outlier_df["building_size_num"] = np.nan
if "construction_year_std" in outlier_df.columns:
    outlier_df["construction_year_num"] = safe_numeric(outlier_df["construction_year_std"])
elif "construction_year" in outlier_df.columns:
    outlier_df["construction_year_num"] = safe_numeric(outlier_df["construction_year"])
else:
    outlier_df["construction_year_num"] = np.nan
# ------------------------------
# 4) Price Base by Regime
# ------------------------------
outlier_df["outlier_price_base"] = np.select(
    [
        outlier_df["price_regime"].eq("sale"),
        outlier_df["price_regime"].eq("rent_only"),
        outlier_df["price_regime"].eq("mortgage_only"),
        outlier_df["price_regime"].eq("mortgage_and_rent"),
    ],
    [
        outlier_df["sale_price_num"],
        outlier_df["monthly_rent_num"],
        outlier_df["deposit_amount_num"],
        outlier_df["deposit_amount_num"] + outlier_df["monthly_rent_num"],
    ],
    default=np.nan
)
# ------------------------------
# 5) Initialize Flags
# ------------------------------
outlier_df["outlier_price_flag"] = False
outlier_df["outlier_area_flag"] = False
outlier_df["outlier_year_flag"] = False
outlier_df["outlier_reason"] = ""
# ------------------------------
# 6) Area Outlier Rules
# ------------------------------
area_bad = (
    outlier_df["building_size_num"].notna()
    & (
        (outlier_df["building_size_num"] <= 5)
        | (outlier_df["building_size_num"] > 2000)
    )
)
outlier_df.loc[area_bad, "outlier_area_flag"] = True
outlier_df.loc[area_bad, "outlier_reason"] += "area_out_of_valid_range; "
# ------------------------------
# 7) Construction Year Outlier Rules
# ------------------------------
year_bad = (
    outlier_df["construction_year_num"].notna()
    & (
        (outlier_df["construction_year_num"] < 1300)
        | (outlier_df["construction_year_num"] > 1405)
    )
)


outlier_df.loc[year_bad, "outlier_year_flag"] = True
outlier_df.loc[year_bad, "outlier_reason"] += "construction_year_out_of_valid_range; "
# ------------------------------
# 8) Price Outlier Rules without Merge
# ------------------------------
group_cols = ["price_regime", "price_status"]
if "city" in outlier_df.columns:
    group_cols = ["city"] + group_cols
valid_price = (
    outlier_df["outlier_price_base"].notna()
    & outlier_df["outlier_price_base"].gt(0)
)
outlier_df["price_iqr_group_count"] = np.nan
outlier_df["price_iqr_q1"] = np.nan
outlier_df["price_iqr_q3"] = np.nan
outlier_df["price_iqr_lower_bound"] = np.nan
outlier_df["price_iqr_upper_bound"] = np.nan

price_work = outlier_df.loc[
    valid_price,
    group_cols + ["outlier_price_base"]
].copy()
if not price_work.empty:
    grouped_price = price_work.groupby(group_cols, dropna=False)["outlier_price_base"]

    price_work["price_iqr_group_count"] = grouped_price.transform("count")
    price_work["price_iqr_q1"] = grouped_price.transform(lambda x: x.quantile(0.25))
    price_work["price_iqr_q3"] = grouped_price.transform(lambda x: x.quantile(0.75))
    price_work["price_iqr_iqr"] = (
        price_work["price_iqr_q3"] - price_work["price_iqr_q1"]
    )
    price_work["price_iqr_lower_bound"] = (
        price_work["price_iqr_q1"] - 3 * price_work["price_iqr_iqr"]
    )
    price_work["price_iqr_upper_bound"] = (
        price_work["price_iqr_q3"] + 3 * price_work["price_iqr_iqr"]
    )
    assign_cols = [
        "price_iqr_group_count",
        "price_iqr_q1",
        "price_iqr_q3",
        "price_iqr_lower_bound",
        "price_iqr_upper_bound",
    ]
    outlier_df.loc[price_work.index, assign_cols] = price_work[assign_cols]
price_bad = (
    valid_price
    & outlier_df["price_iqr_group_count"].ge(30)
    & (
        (outlier_df["outlier_price_base"] < outlier_df["price_iqr_lower_bound"])
        | (outlier_df["outlier_price_base"] > outlier_df["price_iqr_upper_bound"])
    )
)
outlier_df.loc[price_bad, "outlier_price_flag"] = True
outlier_df.loc[price_bad, "outlier_reason"] += "price_outlier_by_regime_iqr; "
# ------------------------------
# 9) Final Outlier Action
# ------------------------------
outlier_df["has_any_outlier_flag"] = (
    outlier_df["outlier_price_flag"]
    | outlier_df["outlier_area_flag"]
    | outlier_df["outlier_year_flag"]
)
outlier_df["outlier_action"] = np.where(
    outlier_df["has_any_outlier_flag"],
    "flag_for_review",
    "keep"
)
outlier_df["outlier_rule_version"] = "outlier_policy_v1_regime_iqr_safe"


outlier_df["outlier_reason"] = (
    outlier_df["outlier_reason"]
    .str.strip()
    .str.rstrip(";")
)
# ------------------------------
# 10) Summary Audit
# ------------------------------
total_records = len(outlier_df)
outlier_summary = pd.DataFrame({
    "metric": [
        "total_records",
        "price_outliers",
        "area_outliers",
        "year_outliers",
        "any_outlier",
        "kept_records",
    ],
    "count": [
        total_records,
        int(outlier_df["outlier_price_flag"].sum()),
        int(outlier_df["outlier_area_flag"].sum()),
        int(outlier_df["outlier_year_flag"].sum()),
        int(outlier_df["has_any_outlier_flag"].sum()),
        int((outlier_df["outlier_action"] == "keep").sum()),
    ],
})
outlier_summary["rate"] = (
    outlier_summary["count"] / total_records
).round(4)
# ------------------------------
# 11) Regime Audit with Rates
# ------------------------------
outlier_regime_audit_final = (
    outlier_df
    .groupby(["price_regime", "price_status"], dropna=False)
    .agg(
        records=("price_regime", "size"),
        price_outliers=("outlier_price_flag", "sum"),
        area_outliers=("outlier_area_flag", "sum"),
        year_outliers=("outlier_year_flag", "sum"),
        any_outlier=("has_any_outlier_flag", "sum"),
    )
    .reset_index()
)
for col in ["price_outliers", "area_outliers", "year_outliers", "any_outlier"]:
    outlier_regime_audit_final[f"{col}_rate"] = (
        outlier_regime_audit_final[col]
        / outlier_regime_audit_final["records"]
    ).round(4)
# ------------------------------
# 12) Clean Numeric Profile
# ------------------------------
profile_cols = [
    "outlier_price_base",
    "building_size_num",
    "construction_year_num",
]
outlier_numeric_profile_clean = (
    outlier_df
    .groupby(["price_regime", "price_status"], dropna=False)[profile_cols]
    .agg(["count", "min", "median", "max"])
    .reset_index()
)
outlier_numeric_profile_clean.columns = [
    "_".join([str(x) for x in col if x != ""]).strip("_")
    for col in outlier_numeric_profile_clean.columns
]
# ------------------------------
# 13) Save Final Outputs
# ------------------------------
outlier_summary.to_csv(
    OUTPUT_TABLES / "outlier_policy_summary.csv",
    index=False,
    encoding="utf-8-sig"
)
outlier_regime_audit_final.to_csv(
    OUTPUT_TABLES / "outlier_policy_regime_audit_final.csv",
    index=False,
    encoding="utf-8-sig"
)
outlier_numeric_profile_clean.to_csv(
    OUTPUT_TABLES / "outlier_policy_numeric_profile_clean.csv",
    index=False,
    encoding="utf-8-sig"
)
outlier_df.to_csv(
    OUTPUT_TABLES / "real_estate_ads_with_outlier_flags.csv",
    index=False,
    encoding="utf-8-sig"
)
df = outlier_df.copy()
print("Outlier Policy completed successfully.")
print("Saved files:")
print(OUTPUT_TABLES / "outlier_policy_summary.csv")
print(OUTPUT_TABLES / "outlier_policy_regime_audit_final.csv")
print(OUTPUT_TABLES / "outlier_policy_numeric_profile_clean.csv")
print(OUTPUT_TABLES / "real_estate_ads_with_outlier_flags.csv")
display(outlier_summary)
display(outlier_regime_audit_final)
display(outlier_numeric_profile_clean)


# ==============================
# Validate Large Outlier Output Safely
# ==============================
import pandas as pd
from pathlib import Path
BASE_DIR = Path(r"C:\Users\Abbas\Desktop\housing_market_project")
OUTPUT_TABLES = BASE_DIR / "outputs" / "tables"
DATA_SAMPLES = BASE_DIR / "data" / "samples"
OUTPUT_TABLES.mkdir(parents=True, exist_ok=True)
DATA_SAMPLES.mkdir(parents=True, exist_ok=True)
final_file = OUTPUT_TABLES / "real_estate_ads_with_outlier_flags.csv"
if not final_file.exists():
    raise FileNotFoundError(f"File not found: {final_file}")
required_cols = [
    "outlier_price_flag",
    "outlier_area_flag",
    "outlier_year_flag",
    "has_any_outlier_flag",
    "outlier_reason",
    "outlier_action",
    "outlier_rule_version",
]

# 1) Read only header
schema_cols = pd.read_csv(final_file, nrows=0).columns.tolist()

missing_cols = [c for c in required_cols if c not in schema_cols]

if missing_cols:
    raise ValueError(f"Missing required outlier columns: {missing_cols}")

# 2) Chunk-based audit
total_rows = 0

action_counts = {}

flag_counts = {
    "outlier_price_flag": 0,
    "outlier_area_flag": 0,
    "outlier_year_flag": 0,
    "has_any_outlier_flag": 0,
}

usecols = [
    "outlier_price_flag",
    "outlier_area_flag",
    "outlier_year_flag",
    "has_any_outlier_flag",
    "outlier_action",
]

for chunk in pd.read_csv(final_file, usecols=usecols, chunksize=100_000):
    total_rows += len(chunk)

    for col in flag_counts:
        flag_counts[col] += chunk[col].astype(bool).sum()

    for action, count in chunk["outlier_action"].value_counts(dropna=False).items():
        action_counts[action] = action_counts.get(action, 0) + count

validation_summary = pd.DataFrame({
    "metric": [
        "total_rows",
        "outlier_price_flag",
        "outlier_area_flag",
        "outlier_year_flag",
        "has_any_outlier_flag",
        "keep",
        "flag_for_review",
    ],
    "count": [
        total_rows,
        int(flag_counts["outlier_price_flag"]),
        int(flag_counts["outlier_area_flag"]),
        int(flag_counts["outlier_year_flag"]),
        int(flag_counts["has_any_outlier_flag"]),
        int(action_counts.get("keep", 0)),
        int(action_counts.get("flag_for_review", 0)),
    ],
})

validation_summary["rate"] = (
    validation_summary["count"] / total_rows
).round(4)

# 3) Save schema, validation summary, and 1000-row sample
schema_df = pd.DataFrame({
    "column_name": schema_cols
})

sample_df = pd.read_csv(final_file, nrows=1000)

if len(sample_df) != 1000:
    raise ValueError(f"Sample rows must be 1000, but got {len(sample_df)} rows.")

validation_summary.to_csv(
    OUTPUT_TABLES / "outlier_final_file_validation_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

schema_df.to_csv(
    OUTPUT_TABLES / "outlier_final_file_schema.csv",
    index=False,
    encoding="utf-8-sig"
)

sample_file = DATA_SAMPLES / "outlier_final_file_sample_1000.csv"

sample_df.to_csv(
    sample_file,
    index=False,
    encoding="utf-8-sig"
)

print("Large outlier file validation completed successfully.")
print("Main file:")
print(final_file)

print("\nSaved audit files:")
print(OUTPUT_TABLES / "outlier_final_file_validation_summary.csv")
print(OUTPUT_TABLES / "outlier_final_file_schema.csv")

print("\nSaved sample file:")
print(sample_file)

print("\nSample check:")
print(f"Rows: {len(sample_df):,}")
print(f"Columns: {len(sample_df.columns):,}")

display(validation_summary)
display(schema_df.tail(15))